# PFC · Linhas de base com Saída Estruturada (Colab)

Roda o **mesmo** `pfc-avaliar`, na **mesma** GPU T4 das rodadas completas, com as linhas de base usadas para medir o que a troca da Saída Estruturada pelo Tool Calling rendeu:

1. **Saída Estruturada** com os quatro modelos locais (Qwen 3 4B, Gemma 4 E4B, Gemma 4 E2B e Mistral Nemo 12B): mesmo prompt, mesmo schema, mesma data de referência, mas a resposta é forçada ao JSON dos parâmetros, sem ferramenta e sem possibilidade de recusa.
2. **Método do protótipo** com o Phi-4 14B: dicionário de normalização, prompt com exemplos, schema com campo `reasoning`, novas tentativas e fallback por regex, como no protótipo do 1º CGEO.

Uma repetição por configuração (temperatura zero). **Antes de começar:** `Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`, depois `Executar tudo`. A célula 3 pede o arquivo `pfc_busca_colab.zip`. A rodada leva cerca de 1 h 30 min; deixe a aba aberta.

> Nenhum serviço de LLM em nuvem é usado: o Ollama roda dentro desta máquina virtual. Só a GPU é alugada.

**Se a sessão cair no meio:** `Executar tudo` de novo; o avaliador pula o que já foi feito enquanto a pasta `results/` existir. A célula 5 baixa um zip parcial ao fim de cada modelo, para não perder o que já rodou.


In [ ]:
# 1) Configuração — NÃO altere depois de começar
MODELOS_SE = ['qwen3:4b-instruct-2507-q4_K_M', 'gemma4:e4b-it-qat', 'gemma4:e2b-it-qat', 'mistral-nemo:12b']
MODELO_PROTOTIPO = 'phi4:14b'
HOJE = '2026-09-14'     # a mesma data de referência das rodadas completas com Tool Calling
DATASET_SHA256 = '07bf424a40d68392cad4a190c334b3b6144676d2927a75ae6b0b5328abf0ba42'   # gravado por scripts/empacotar_colab.py
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
# 2) Instala o Ollama (precisa de zstd) e sobe o servidor
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y zstd pciutils > /dev/null 2>&1 && echo "zstd instalado"
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

import os, shutil, subprocess, time, json, urllib.request
assert shutil.which('ollama'), 'o Ollama NÃO foi instalado — veja as mensagens acima'
servidor = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT,
                            env=dict(os.environ, OLLAMA_KEEP_ALIVE='30m'))
for _ in range(40):
    try:
        v = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3))['version']
        print('Ollama', v, 'no ar')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('o servidor do Ollama não subiu — veja /content/ollama.log')

In [ ]:
# 3) Envia o repositório (pfc_busca_colab.zip), confere o dataset e instala o pacote
import os, hashlib, shutil, datetime
from google.colab import files

def sha_dataset():
    p = '/content/pfc_busca/data/dataset.json'
    return hashlib.sha256(open(p, 'rb').read()).hexdigest() if os.path.exists(p) else None

if os.path.exists('/content/pfc_busca') and sha_dataset() != DATASET_SHA256:
    antigo = '/content/pfc_busca_anterior_' + datetime.datetime.now().strftime('%H%M%S')
    shutil.move('/content/pfc_busca', antigo)
    print('pacote anterior movido para', antigo, '(não será usado)')
if not os.path.exists('/content/pfc_busca/pyproject.toml'):
    enviado = files.upload()
    nome = next(iter(enviado))
    !unzip -q -o "$nome" -d /content
assert sha_dataset() == DATASET_SHA256, (
    'O zip enviado NÃO é o pacote atual (o hash do dataset não confere). Baixe de novo dist/pfc_busca_colab.zip.')
print('dataset conferido:', DATASET_SHA256[:12])
%cd /content/pfc_busca
!pip install -q -e . 2>&1 | tail -2
!python -c "from pfc_busca.evaluation.dataset_builder import carregar_dataset; print(len(carregar_dataset()), 'consultas no dataset')"


In [ ]:
# 4) Baixa os modelos (~28 GB; 10-15 min)
for m in MODELOS_SE + [MODELO_PROTOTIPO]:
    print(m, end=': ', flush=True)
    !ollama pull {m} 2>&1 | tail -1
!ollama list


In [ ]:
# 5) Linhas de base: Saída Estruturada (4 modelos) e método do protótipo (Phi-4 14B), 310 consultas, 1 repetição
import datetime, re, subprocess
from google.colab import files

def rodar(modelo, abordagem):
    cmd = ['pfc-avaliar', '--modelo', modelo, '--abordagem', abordagem, '--repeticoes', '1', '--hoje', HOJE, '--sem-sql']
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, COLUMNS='110'))
    n = 0
    for linha in proc.stdout:
        linha = linha.rstrip()
        if re.match(r'^[✓·×] r\d', linha):
            n += 1
            if linha.startswith('×') or n % 50 == 0:
                print(f'   [{n}] {linha}', flush=True)
        elif linha.strip() and not linha.startswith('avaliando'):
            print(linha, flush=True)
    return proc.wait()

def baixar_parcial():
    !cd /content && rm -f linha_de_base_colab.zip && zip -q -r linha_de_base_colab.zip pfc_busca/results/se-* pfc_busca/results/prototipo-* 2>/dev/null; ls -la linha_de_base_colab.zip

rodadas = [(m, 'saida_estruturada') for m in MODELOS_SE] + [(MODELO_PROTOTIPO, 'prototipo')]
for modelo, abordagem in rodadas:
    print(f'\n== {abordagem} · {modelo} == início {datetime.datetime.now():%H:%M}', flush=True)
    codigo = rodar(modelo, abordagem)
    print(f'   saída {codigo} às {datetime.datetime.now():%H:%M}', flush=True)
    subprocess.run(['ollama', 'stop', modelo], capture_output=True)
    baixar_parcial()


In [ ]:
# 6) Baixa o resultado final
baixar_parcial()
files.download('/content/linha_de_base_colab.zip')


## De volta ao PC

Descompacte `linha_de_base_colab.zip` na pasta que contém `pfc_busca/` (as rodadas vão para `pfc_busca/results/se-*` e `pfc_busca/results/prototipo-*`) e regenere as tabelas do texto com `python scripts/gerar_tabelas_texto.py --texto texto`.
